# Rally E2B Export Prep

Prepares a reusable Kaggle kernel source with the exact `heretic-to-onnx` checkout and the optimized Gemma4 q4f16 template needed by the export lanes.

In [ ]:
import os, platform, shutil
from pathlib import Path

print('python_platform=', platform.platform())
print('working_disk_free_gb=', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))
print('input_dirs=', [str(p) for p in Path('/kaggle/input').glob('*')])


In [ ]:
import os, subprocess, sys

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
packages = [
    'huggingface_hub[cli]>=1.5.0',
    'hf_transfer>=0.1.9',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])


In [ ]:
import json, os, subprocess
from pathlib import Path
from huggingface_hub import snapshot_download

REPO_URL = os.environ.get('HERETIC_TO_ONNX_REPO', 'https://github.com/alkahest-ai/heretic-to-onnx.git')
REPO_REF = os.environ.get('HERETIC_TO_ONNX_REF', 'codex/kaggle-heretic-2b-run')
TEMPLATE_MODEL_ID = os.environ.get('RALLY_OPTIMIZED_TEMPLATE_MODEL_ID', 'onnx-community/gemma-4-E2B-it-ONNX')
TEMPLATE_REVISION = os.environ.get('RALLY_OPTIMIZED_TEMPLATE_REVISION', '')
STAGE_ROOT = Path('/kaggle/working/rally-e2b-export-prep')
REPO_DIR = STAGE_ROOT / 'heretic-to-onnx'
TEMPLATE_DIR = STAGE_ROOT / 'optimized-template'
STAGE_ROOT.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', REPO_REF, '--depth', '1', REPO_URL, str(REPO_DIR)])

kwargs = {
    'repo_id': TEMPLATE_MODEL_ID,
    'local_dir': str(TEMPLATE_DIR),
    'allow_patterns': ['onnx/decoder_model_merged_q4f16.onnx'],
}
if TEMPLATE_REVISION:
    kwargs['revision'] = TEMPLATE_REVISION
snapshot_download(**kwargs)

report = {
    'ok': True,
    'repo_dir': str(REPO_DIR),
    'repo_head': subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip(),
    'template_dir': str(TEMPLATE_DIR),
    'template_model_id': TEMPLATE_MODEL_ID,
    'template_revision': TEMPLATE_REVISION,
    'template_files': sorted(str(path.relative_to(TEMPLATE_DIR)) for path in TEMPLATE_DIR.rglob('*') if path.is_file()),
}
report_path = STAGE_ROOT / 'rally-e2b-export-prep-report.json'
report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps(report, indent=2, sort_keys=True))
